In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import joblib
import warnings
import matplotlib.pyplot as plt
import time
from functools import partial  # <-- IMPORT PENTING UNTUK KERNEL ANOVA

warnings.filterwarnings("ignore")

# ==========================================
# 1. LOAD DATA & SCALING
# ==========================================
base_path = './split/'
fold_path = './split/folds/'

print(">>> Memuat data 10-Fold dan menginisiasi MinMaxScaler...")
# Load 10 Fold
folds_data = []
for i in range(1, 11):
    tr_df = pd.read_csv(f"{fold_path}fold_{i}_train.csv")
    val_df = pd.read_csv(f"{fold_path}fold_{i}_val.csv")

    X_tr = tr_df.drop(columns=['Produksi']).values
    y_tr_asli = tr_df['Produksi'].values.reshape(-1, 1)

    X_val = val_df.drop(columns=['Produksi']).values
    y_val_asli = val_df['Produksi'].values.reshape(-1, 1)

    # Normalisasi target Produksi dengan MinMaxScaler per fold
    scaler_y_fold = MinMaxScaler()
    y_tr_scaled = scaler_y_fold.fit_transform(y_tr_asli).ravel()
    
    # y_val_scaled disiapkan, namun evaluasi murni menggunakan y_val_asli
    y_val_scaled = scaler_y_fold.transform(y_val_asli).ravel() 

    folds_data.append((X_tr, y_tr_scaled, X_val, y_val_scaled, scaler_y_fold, y_val_asli.ravel()))

# 5 Fold pertama untuk alokasi pencarian PSO
folds_data_pso = folds_data[:5]

# Data Final train-test
train_final = pd.read_csv(base_path + '90training_siap_final.csv')
test_final = pd.read_csv(base_path + '10testing_siap_final.csv')

X_train_final = train_final.drop(columns=['Produksi']).values
y_train_final_asli = train_final['Produksi'].values.reshape(-1, 1)

X_test_final = test_final.drop(columns=['Produksi']).values
y_test_final_asli = test_final['Produksi'].values.reshape(-1, 1)

# Scaler global untuk model final
scaler_y_final = MinMaxScaler()
y_train_final_scaled = scaler_y_final.fit_transform(y_train_final_asli).ravel()
y_test_final_scaled = scaler_y_final.transform(y_test_final_asli).ravel()

# Data asli untuk keperluan ekspor tabel CSV
test_asli = pd.read_csv(base_path + '9testing.csv')

bulan_map = {
    'Januari':1, 'Februari':2, 'Maret':3, 'April':4,
    'Mei':5, 'Juni':6, 'Juli':7, 'Agustus':8,
    'September':9, 'Oktober':10, 'November':11, 'Desember':12
}

if 'Periode' in test_asli.columns:
    test_asli[['Nama_Bulan', 'Tahun']] = test_asli['Periode'].str.split(' ', expand=True)
    test_asli['Bulan'] = test_asli['Nama_Bulan'].map(bulan_map)
    test_asli['Tahun'] = test_asli['Tahun'].astype(int)
    test_asli.drop(columns=['Periode', 'Nama_Bulan'], inplace=True)


# ==========================================
# 2. FUNGSI KERNEL ANOVA RBF
# ==========================================
def hitung_anova_rbf(X, Y, gamma, degree=2):
    X = np.asarray(X)
    Y = np.asarray(Y)
    K = np.zeros((X.shape[0], Y.shape[0]))
    
    for k in range(X.shape[1]):
        diff_sq = (X[:, k].reshape(-1, 1) - Y[:, k].reshape(1, -1)) ** 2
        K += np.exp(-gamma * diff_sq)
        
    K = K / X.shape[1] 
    
    return K ** degree


# ==========================================
# 3. FITNESS PSO (5 FOLD)
# ==========================================
def evaluate_svr_cv(params):
    C, epsilon, gamma = params
    rmse_scores = []

    try:
        # Loop hanya pada 5 fold pertama
        for X_tr, y_tr_scaled, X_val, y_val_scaled, scaler_y_fold, y_val_asli in folds_data_pso:
            
            # Gunakan partial untuk menitipkan nilai gamma ke fungsi kernel
            kernel_func = partial(hitung_anova_rbf, gamma=gamma)

            model = SVR(
                kernel=kernel_func,
                C=C,
                epsilon=epsilon,
                max_iter=10000 
            )

            # Training menggunakan target yang dinormalisasi (0-1)
            model.fit(X_tr, y_tr_scaled)

            # Prediksi masih dalam skala 0-1
            y_pred_scaled = model.predict(X_val)

            # Denormalisasi agar error dihitung dalam satuan aslinya (Ton)
            y_pred_asli = scaler_y_fold.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

            # Mencegah adanya nilai prediksi minus
            y_pred_asli = np.clip(y_pred_asli, 0, None)

            rmse = np.sqrt(mean_squared_error(y_val_asli, y_pred_asli))
            rmse_scores.append(rmse)

        return np.mean(rmse_scores)

    except Exception as e:
        return float('inf')


# ==========================================
# 4. ALGORITMA PSO UTAMA
# ==========================================
def pso_auto_resume(n_particles, target_iter, timeout=60): # Timeout dibuat lebih besar untuk komputasi kustom matriks ANOVA

    np.random.seed(42)

    # Nama file state disesuaikan
    checkpoint = f"pso_state_rmse_anova90_fold_{n_particles}_partikel.save"

    # Rentang parameter disesuaikan dengan script ANOVA asli Anda
    lb = np.array([1, 0.000001, 0.00001])
    ub = np.array([1000, 0.1, 100])

    w, c1, c2 = 0.7, 1.5, 1.5

    if os.path.exists(checkpoint):
        print(f"\n>>> File checkpoint ditemukan! Memuat data {n_particles} partikel...")
        state = joblib.load(checkpoint)

        start_iter = state['iterasi_terakhir']
        particles = state['particles']
        velocities = state['velocities']
        personal_best = state['personal_best']
        personal_best_score = state['personal_best_score']
        global_best = state['global_best']
        global_best_score = state['global_best_score']
        rmse_history = state['rmse_history']
        
        print(f">>> Melanjutkan dari iterasi ke-{start_iter + 1} menuju {target_iter}...\n")
    else:
        print(f"\n>>> Memulai PSO dari awal untuk {n_particles} partikel (Kernel ANOVA)...\n")
        start_iter = 0
        particles = np.random.uniform(lb, ub, (n_particles, 3))
        velocities = np.zeros((n_particles, 3))
        personal_best = particles.copy()
        personal_best_score = np.array([float('inf')] * n_particles)
        global_best = None
        global_best_score = float('inf')
        rmse_history = []

    if start_iter >= target_iter:
        print("Target iterasi sudah tercapai di eksekusi sebelumnya.")
        return rmse_history

    failed_count = 0

    for i in range(start_iter, target_iter):
        print(f"Iterasi ke {i+1}/{target_iter} :")
        iter_success = False

        for j in range(n_particles):
            start_time = time.time()
            
            score = evaluate_svr_cv(particles[j])
            
            elapsed = time.time() - start_time
            
            # Jika memakan waktu melebihi batas timeout
            if elapsed > timeout:
                score = float('inf')

            if score < personal_best_score[j]:
                personal_best[j] = particles[j]
                personal_best_score[j] = score

                if score < global_best_score:
                    global_best = particles[j].copy()
                    global_best_score = score
                    iter_success = True

            score_tampil = f"{score:.4f}" if score != float('inf') else "inf"
            print(f"partikel {j+1}/{n_particles}, RMSE : {score_tampil}, waktu: {elapsed:.4f} s.")

        if not iter_success and global_best_score == float('inf'):
            failed_count += 1
        else:
            failed_count = 0

        rmse_history.append(global_best_score)

        for j in range(n_particles):
            r1, r2 = np.random.rand(), np.random.rand()
            gb = global_best if global_best is not None else personal_best[j]
            
            velocities[j] = (
                w * velocities[j]
                + c1 * r1 * (personal_best[j] - particles[j])
                + c2 * r2 * (gb - particles[j])
            )
            particles[j] += velocities[j]
            particles[j] = np.clip(particles[j], lb, ub)

        print(f"--- Iterasi {i+1} Selesai | Global Best RMSE = {global_best_score:.4f} ---\n")

        state = {
            'iterasi_terakhir': i + 1,
            'particles': particles,
            'velocities': velocities,
            'personal_best': personal_best,
            'personal_best_score': personal_best_score,
            'global_best': global_best,
            'global_best_score': global_best_score,
            'rmse_history': rmse_history
        }

        joblib.dump(state, checkpoint)
        
        if failed_count >= 5:
            print("\n[!] WARNING: 5 iterasi berturut-turut gagal! Cek rentang batas parameter.")

    return rmse_history


# ==========================================
# 5. RUN PROGRAM MAIN
# ==========================================
JUMLAH_PARTIKEL = 100
TARGET_ITERASI = 100
TIMEOUT_DETIK = 60 # Membutuhkan durasi lebih panjang dibanding RBF standar

history = pso_auto_resume(
    n_particles=JUMLAH_PARTIKEL,
    target_iter=TARGET_ITERASI,
    timeout=TIMEOUT_DETIK
)

state = joblib.load(f"pso_state_rmse_anova90_fold_{JUMLAH_PARTIKEL}_partikel.save")

C_best, epsilon_best, gamma_best = state['global_best']

print("\n" + "="*50)
print("HASIL AKHIR PARAMETER")
print("="*50)
print(f"C      = {C_best:.4f}")
print(f"epsilon= {epsilon_best:.6f}")
print(f"gamma  = {gamma_best:.4f}")


# ==========================================
# 6. VALIDASI 10 FOLD DENGAN ANOVA
# ==========================================
print("\n" + "="*50)
print("VALIDASI 10-FOLD CV (ANOVA KERNEL)")
print("="*50)

# Titipkan gamma hasil PSO ke fungsi kernel terbaik
kernel_func_best = partial(hitung_anova_rbf, gamma=gamma_best)

best_model_cv = SVR(
    kernel=kernel_func_best,
    C=C_best,
    epsilon=epsilon_best,
    max_iter=10000
)

cv_rmse_scores = []
fold_terbaik = 1
rmse_terbaik = float('inf')

for i, (X_tr, y_tr_scaled, X_val, y_val_scaled, scaler_y_fold, y_val_asli) in enumerate(folds_data):

    best_model_cv.fit(X_tr, y_tr_scaled)

    y_pred_scaled = best_model_cv.predict(X_val)

    # Denormalisasi evaluasi per-Fold (Ton)
    y_pred_asli = scaler_y_fold.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
    y_pred_asli = np.clip(y_pred_asli, 0, None)

    rmse_fold = np.sqrt(mean_squared_error(y_val_asli, y_pred_asli))
    cv_rmse_scores.append(rmse_fold)

    print(f"Fold {i+1:02d} | RMSE = {rmse_fold:.4f} Ton")

    if rmse_fold < rmse_terbaik:
        rmse_terbaik = rmse_fold
        fold_terbaik = i + 1

print(f"\nFold terbaik   : Fold {fold_terbaik}")
print(f"RMSE terbaik   : {rmse_terbaik:.4f} Ton")
print(f"RMSE rata-rata : {np.mean(cv_rmse_scores):.4f} Ton")
print(f"RMSE Std Dev   : {np.std(cv_rmse_scores):.4f} Ton")


# ==========================================
# 7. FINAL TRAINING & EVALUATION
# ==========================================
# Membangun ulang model agar segar
model_best = SVR(
    kernel=kernel_func_best,
    C=C_best,
    epsilon=epsilon_best,
    max_iter=10000
)

# Training menggunakan target dari global scaler
model_best.fit(X_train_final, y_train_final_scaled)

joblib.dump(model_best, 'model_svr_anova_best_rmse_cv.save')
print('\n✓ Model final tersimpan di: model_svr_anova_best_rmse_cv.save')

# Memprediksi Data Training & Testing Final (Hasilnya berskala 0-1)
y_train_pred_scaled = model_best.predict(X_train_final)
y_test_pred_scaled = model_best.predict(X_test_final)

# Denormalisasi menjadi satuan Ton
y_train_pred_asli = scaler_y_final.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).ravel()
y_test_pred_asli = scaler_y_final.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()

y_train_pred_asli = np.clip(y_train_pred_asli, 0, None)
y_test_pred_asli = np.clip(y_test_pred_asli, 0, None)

y_train_asli = y_train_final_asli.ravel()
y_test_asli = y_test_final_asli.ravel()

print('\n' + '='*50)
print('EVALUASI TRAINING (SKALA ASLI/TON) :')
print('='*50)
print(f"RMSE : {np.sqrt(mean_squared_error(y_train_asli, y_train_pred_asli)):.4f} Ton")
print(f"R2   : {r2_score(y_train_asli, y_train_pred_asli):.4f}")

print('\n' + '='*50)
print('EVALUASI TESTING (SKALA ASLI/TON) :')
print('='*50)
print(f"RMSE : {np.sqrt(mean_squared_error(y_test_asli, y_test_pred_asli)):.4f} Ton")
print(f"R2   : {r2_score(y_test_asli, y_test_pred_asli):.4f}")


# ==========================================
# 8. EXPORT TABEL CSV PREDIKSI
# ==========================================
hasil = test_asli.copy()

hasil['Produksi_Asli'] = y_test_asli.round(2)
hasil['Prediksi_Ton'] = y_test_pred_asli.round(2)

kolom_kab = 'Kabupaten/Kota' if 'Kabupaten/Kota' in hasil.columns else 'Kabupaten'

if 'Tahun' in hasil.columns and 'Bulan' in hasil.columns and kolom_kab in hasil.columns:
    hasil = hasil[['Tahun', 'Bulan', kolom_kab, 'Produksi_Asli', 'Prediksi_Ton']]

nama_file_csv = f'hasil_terbaik_anova_rmse_10fold_{JUMLAH_PARTIKEL}_partikel.csv'
hasil.to_csv(nama_file_csv, index=False)


# ==========================================
# 9. GRAFIK KONVERGENSI
# ==========================================
if len(history) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(history, linewidth=2)
    plt.title(f"Konvergensi PSO ANOVA ({JUMLAH_PARTIKEL} Partikel) - Optimasi 10-Fold CV", fontsize=14)
    plt.xlabel("Iterasi", fontsize=12)
    plt.ylabel("RMSE (Ton)", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\n" + "="*50)
print(">>> SELURUH PROSES SELESAI <<<")
print("="*50)